# 03 – Feature Engineering

Die Features werden in `src/energy_analytics/features.py` erzeugt und sind durch
Unit-Tests abgesichert (`tests/test_pipeline.py`). Dieses Notebook erläutert die
fachliche Bedeutung und prüft die Ergebnisse.

| Gruppe | Features | Hinweis |
|---|---|---|
| Kalender | `year`, `quarter`, `month`, `day`, `hour`, `day_of_week`, `is_weekend`, `season` | in deutscher **Ortszeit** |
| Energie | `wind`, `renewable_generation`, `residual_load`, `renewable_share_pct` | Residuallast = Last − Wind − Solar |
| Preis | `price_lag_1/24/168`, `price_rolling_mean_24h`, `price_rolling_std_24h` | nur **vergangene** Werte |
| Flags | `is_negative_price` | für BI-Auswertungen |

In [1]:
import sys
from pathlib import Path

# Projektpaket importierbar machen (alternativ: pip install -e .)
sys.path.insert(0, str(Path.cwd().parent / "src"))

In [2]:
import pandas as pd

from energy_analytics.database import load_energy_data

df = load_energy_data()
df.head()

,timestamp,timestamp_local,date,price,wind_offshore,wind_onshore,wind,solar,load,year,...,season,renewable_generation,residual_load,renewable_share_pct,price_lag_1,price_lag_24,price_lag_168,price_rolling_mean_24h,price_rolling_std_24h,is_negative_price
0,2022-12-31 23:00:00+00:00,2023-01-01 00:00:00,2023-01-01,-5.17,3060.925,27360.450,30421.375,0.00,38346.050,2023,...,Winter,30421.375,7924.675,79.333791,-0.68,-0.03,120.28,-0.792083,1.122431,True
1,2023-01-01 00:00:00+00:00,2023-01-01 01:00:00,2023-01-01,-1.07,3588.425,27965.775,31554.200,0.00,37777.200,2023,...,Winter,31554.200,6223.000,83.527101,-5.17,-0.07,112.11,-1.006250,1.421285,True
2,2023-01-01 01:00:00+00:00,2023-01-01 02:00:00,2023-01-01,-1.47,3844.575,27897.050,31741.625,0.00,36939.925,2023,...,Winter,31741.625,5198.300,85.927692,-1.07,-0.03,106.65,-1.047917,1.407233,True
3,2023-01-01 02:00:00+00:00,2023-01-01 03:00:00,2023-01-01,-5.08,3465.125,25986.475,29451.600,0.00,35932.375,2023,...,Winter,29451.600,6480.775,81.963967,-1.47,-0.04,90.84,-1.107917,1.392567,True
4,2023-01-01 03:00:00+00:00,2023-01-01 04:00:00,2023-01-01,-4.49,3464.275,25462.150,28926.425,1.05,35486.250,2023,...,Winter,28927.475,6558.775,81.517419,-5.08,-0.03,85.70,-1.317917,1.590477,True


## Residuallast und Anteil erneuerbarer Energien

$$\text{Residuallast}_t = \text{Last}_t - (\text{Wind}_t + \text{Solar}_t)$$

$$\text{EE-Anteil}_t = \frac{\text{Wind}_t + \text{Solar}_t}{\text{Last}_t} \cdot 100$$

Der EE-Anteil kann über 100 % liegen, wenn Wind und Solar die Last übersteigen.
Der Überschuss wird exportiert, gespeichert oder abgeregelt.

In [3]:
df[["renewable_generation", "residual_load", "renewable_share_pct"]].describe().T.round(1)

,count,mean,std,min,25%,50%,75%,max
renewable_generation,26304.0,22455.3,13157.0,127.6,11252.9,21153.0,32260.7,63658.0
residual_load,26304.0,30377.1,13701.3,-6406.3,20833.7,30695.1,39813.3,68313.1
renewable_share_pct,26304.0,42.4,24.2,0.2,22.4,40.3,59.9,115.2


In [4]:
over = df[df["renewable_share_pct"] > 100]
print(f"Stunden mit EE-Anteil > 100 %: {len(over)} ({len(over) / len(df):.2%})")
print(f"Ø Preis in diesen Stunden:     {over['price'].mean():.2f} EUR/MWh")
print(f"davon negative Preise:         {(over['price'] < 0).mean():.0%}")

Stunden mit EE-Anteil > 100 %: 186 (0.71%)
Ø Preis in diesen Stunden:     -28.03 EUR/MWh
davon negative Preise:         89%


## Vermeidung von Look-Ahead-Bias

Rollierende Kennzahlen werden auf `price.shift(1)` berechnet. Zum Zeitpunkt *t*
enthält `price_rolling_mean_24h` also den Durchschnitt der Stunden *t−24 … t−1*,
nicht den aktuellen Preis. Die folgende Prüfung bestätigt das:

In [5]:
i = 1000
manual = df.loc[i - 24 : i - 1, "price"].mean()
print(f"Feature:  {df.loc[i, 'price_rolling_mean_24h']:.4f}")
print(f"Manuell:  {manual:.4f}")
assert abs(df.loc[i, "price_rolling_mean_24h"] - manual) < 1e-9

Feature:  95.0721
Manuell:  95.0721


Da die Pipeline 8 Tage Vorlauf abruft, sind auch die Lag-Features der ersten
Januarwoche 2023 vollständig – vorher mussten diese Zeilen verworfen werden:

In [6]:
df.filter(like="price_").isna().sum()

price_lag_1               0
price_lag_24              0
price_lag_168             0
price_rolling_mean_24h    0
price_rolling_std_24h     0
dtype: int64

## Multikollinearität

`renewable_generation`, `residual_load` und `renewable_share_pct` sind
mathematisch voneinander abhängig. Für Baum-Modelle ist das unkritisch, für
lineare Modelle sollte man die Koeffizienten jedoch nicht einzeln interpretieren.

In [7]:
df[["wind", "solar", "load", "renewable_generation", "residual_load", "renewable_share_pct"]].corr().round(2)

,wind,solar,load,renewable_generation,residual_load,renewable_share_pct
wind,1.00,-0.26,0.16,0.62,-0.49,0.60
solar,-0.26,1.00,0.19,0.59,-0.44,0.56
load,0.16,0.19,1.00,0.29,0.39,0.02
renewable_generation,0.62,0.59,0.29,1.00,-0.77,0.95
residual_load,-0.49,-0.44,0.39,-0.77,1.00,-0.90
renewable_share_pct,0.60,0.56,0.02,0.95,-0.90,1.00
